
# Bootstrap and Permutation Methods

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Bootstrap methods estimate uncertainty by resampling observations with replacement. Permutation tests estimate a null distribution by shuffling labels when the null says labels are exchangeable.

**Highest-probability exam pattern:** Given two groups or one fitted statistic, compute a bootstrap confidence interval or a permutation-test p-value without relying on asymptotic formulas.



## Assumptions, intuition, and theory

Bootstrap resampling approximates the sampling distribution of a statistic using the empirical distribution. Permutation tests require exchangeability under the null hypothesis; they are especially natural for two-sample mean differences and randomized experiments.



    ## Python method notes and documentation

    `np.random.default_rng` controls resampling, `np.quantile` forms percentile intervals, and a permutation p-value is the fraction of shuffled null statistics at least as extreme as the observed statistic.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import arrays and random-number generation.
import pandas as pd  # Import DataFrame tables for bootstrap and permutation summaries.
import matplotlib.pyplot as plt  # Import plotting tools for null and bootstrap distributions.
from scipy import stats  # Import statistical helper functions for reference calculations.
RANDOM_SEED = 123  # Store the reproducibility seed in one visible variable.
rng = np.random.default_rng(RANDOM_SEED)  # Create one random generator for all resampling.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def bootstrap_ci(data, statistic_function=np.mean, n_boot=2000, confidence=0.95, random_state=123):  # Define a percentile bootstrap CI helper.
    rng = np.random.default_rng(random_state)  # Create a reproducible generator inside the function.
    data = np.asarray(data)  # Convert input data to a NumPy array.
    n = len(data)  # Store the original sample size.
    bootstrap_statistics = np.empty(n_boot)  # Allocate space for bootstrap statistic values.
    for b in range(n_boot):  # Repeat the resampling process many times.
        bootstrap_sample = data[rng.integers(0, n, size=n)]  # Sample n observations with replacement.
        bootstrap_statistics[b] = statistic_function(bootstrap_sample)  # Compute the statistic on the resample.
    alpha = 1.0 - confidence  # Convert confidence level to tail probability.
    lower, upper = np.quantile(bootstrap_statistics, [alpha / 2.0, 1.0 - alpha / 2.0])  # Use percentile endpoints.
    estimate = statistic_function(data)  # Compute the statistic on the original sample.
    return estimate, lower, upper, bootstrap_statistics  # Return point estimate, interval, and simulated distribution.

def permutation_test_two_means(group_a, group_b, n_perm=5000, alternative='two-sided', random_state=123):  # Define a two-sample permutation test.
    rng = np.random.default_rng(random_state)  # Create a reproducible generator inside the function.
    group_a = np.asarray(group_a)  # Convert group A to an array.
    group_b = np.asarray(group_b)  # Convert group B to an array.
    observed_difference = np.mean(group_a) - np.mean(group_b)  # Compute the observed difference in sample means.
    combined = np.concatenate([group_a, group_b])  # Combine outcomes under the null hypothesis of exchangeable labels.
    n_a = len(group_a)  # Store the size of the first group.
    null_statistics = np.empty(n_perm)  # Allocate space for null differences.
    for i in range(n_perm):  # Repeat random label shuffling many times.
        shuffled = rng.permutation(combined)  # Randomly permute the pooled outcomes.
        null_statistics[i] = np.mean(shuffled[:n_a]) - np.mean(shuffled[n_a:])  # Recompute the difference after relabeling.
    if alternative == 'two-sided':  # Use a two-sided alternative when direction is not prespecified.
        p_value = np.mean(np.abs(null_statistics) >= abs(observed_difference))  # Count null results at least as extreme in magnitude.
    elif alternative == 'greater':  # Use a greater-than alternative when group A is hypothesized larger.
        p_value = np.mean(null_statistics >= observed_difference)  # Count null results at least as large.
    elif alternative == 'less':  # Use a less-than alternative when group A is hypothesized smaller.
        p_value = np.mean(null_statistics <= observed_difference)  # Count null results at least as small.
    else:  # Guard against misspelled alternatives.
        raise ValueError("alternative must be 'two-sided', 'greater', or 'less'")  # Explain the accepted options.
    return observed_difference, p_value, null_statistics  # Return observed statistic, p-value, and null distribution.


In [ ]:
group_a = rng.normal(loc=1.0, scale=1.0, size=35)  # Simulate measurements from treatment A.
group_b = rng.normal(loc=0.35, scale=1.0, size=35)  # Simulate measurements from treatment B.
estimate, lower, upper, bootstrap_statistics = bootstrap_ci(group_a, np.mean, n_boot=2000, random_state=RANDOM_SEED)  # Bootstrap the mean of group A.
observed_difference, p_value, null_statistics = permutation_test_two_means(group_a, group_b, n_perm=4000, random_state=RANDOM_SEED)  # Run the permutation test.
summary = pd.DataFrame(  # Build a compact result table.
    [  # Start the row list.
        {'quantity': 'mean of group A', 'estimate': estimate, 'lower': lower, 'upper': upper, 'p_value': np.nan},  # Add the bootstrap CI row.
        {'quantity': 'mean(A) - mean(B)', 'estimate': observed_difference, 'lower': np.nan, 'upper': np.nan, 'p_value': p_value},  # Add the permutation row.
    ]  # End the row list.
)  # Finish constructing the table.
display(summary)  # Display the table.
plt.figure(figsize=(6, 4))  # Create a figure for the bootstrap distribution.
plt.hist(bootstrap_statistics, bins=30, edgecolor='black', alpha=0.75)  # Plot bootstrap means.
plt.axvline(lower, color='red', linestyle='--', label='CI lower')  # Mark the lower CI endpoint.
plt.axvline(upper, color='red', linestyle='--', label='CI upper')  # Mark the upper CI endpoint.
plt.xlabel('bootstrap mean')  # Label the x-axis.
plt.ylabel('count')  # Label the y-axis.
plt.title('Bootstrap distribution for group A mean')  # Title the plot.
plt.legend()  # Show the CI labels.
plt.show()  # Render the bootstrap plot.
plt.figure(figsize=(6, 4))  # Create a figure for the permutation null distribution.
plt.hist(null_statistics, bins=30, edgecolor='black', alpha=0.75)  # Plot the null differences.
plt.axvline(observed_difference, color='red', linestyle='--', label='observed difference')  # Mark the observed statistic.
plt.xlabel('permuted mean difference')  # Label the x-axis.
plt.ylabel('count')  # Label the y-axis.
plt.title('Permutation null distribution')  # Title the plot.
plt.legend()  # Show the observed-statistic label.
plt.show()  # Render the permutation plot.



## Exam-style problem prompt

A study reports two independent groups and asks whether the treatment changed the mean outcome. Use a permutation test for the p-value and a bootstrap interval for the effect size or one group mean.



    ## Adaptable code pattern

    ```python
    # Step 1: Define the statistic, such as mean(A) - mean(B).
# Step 2: For bootstrap, resample observations with replacement and recompute the statistic.
# Step 3: Use quantiles of bootstrap statistics for a confidence interval.
# Step 4: For permutation, pool the groups and repeatedly shuffle labels.
# Step 5: Compare the observed statistic to the shuffled null distribution.
# Step 6: Interpret the interval or p-value in the language of the problem.
    ```



## What to remember for the exam

Bootstrap answers “how uncertain is this statistic?” Permutation answers “how surprising is this statistic if labels do not matter?” Keep those two goals separate in your exam explanation.
